# Best Tacklers Leaderboard Analysis

This notebook processes player data to generate a comprehensive best tacklers leaderboard using machine learning techniques and team context factors.

## Section 1: Import Required Libraries

Import pandas, numpy, and scikit-learn RandomForestRegressor for data processing and model training.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import os

# Set the working directory to player_stats folder
os.chdir(r'c:\Users\vardh\Desktop\MY.PROJECTS\prediction.oc\Prediction.oc\player_stats')

print("=== PROCESSING BEST TACKLERS LEADERBOARD ===")
print("Libraries imported successfully!")

=== PROCESSING BEST TACKLERS LEADERBOARD ===
Libraries imported successfully!


## Section 2: Load All Data Files

Load all CSV files including cleaned_players_data, player_stats, shooting data, goals, players, squads, tournaments, standings, and team position probabilities.

In [2]:
# Load all files to ensure full dataset usage
df_data = pd.read_csv('cleaned_players_data-2024_2025.csv')
df_stats = pd.read_csv('cleaned_player_stats.csv')
df_shooting = pd.read_csv('cleaned_player_shooting.csv')
df_goals = pd.read_csv('goals_cleaned.csv')
df_players = pd.read_csv('cleaned_players.csv')
df_squads = pd.read_csv('cleaned_squads.csv')
df_tournaments = pd.read_csv('cleaned_tournaments.csv')
df_standings = pd.read_csv('cleaned_predicted_group_standings.csv')
df_probs = pd.read_csv('cleaned_team_position_probabilities.csv')

print("All data files loaded successfully!")
print(f"Main data shape: {df_data.shape}")
print(f"Standings data shape: {df_standings.shape}")
print(f"Team probabilities shape: {df_probs.shape}")

All data files loaded successfully!
Main data shape: (2854, 267)
Standings data shape: (48, 4)
Team probabilities shape: (48, 6)


## Section 3: Data Preprocessing and Cleaning

Fill missing values for tackle-related columns (tkl, tklw, tkl%, tkl+int, min, blocks_stats_defense, int) and create team multipliers based on qualification probability and points.

In [3]:
# Fill missing values for baseline features
tackle_columns = ['tkl', 'tklw', 'tkl%', 'tkl+int', 'min', 'blocks_stats_defense', 'int']

for col in tackle_columns:
    if col in df_data.columns:
        df_data[col] = df_data[col].fillna(0)
        print(f"Filled missing values for {col}")

# Incorporate context from team-level probability and standings maps
prob_dict = dict(zip(df_probs['team'], df_probs['qualify']))
points_dict = df_standings.groupby('team')['points'].max().to_dict()

def get_team_multiplier(squad_name):
    prob = prob_dict.get(squad_name, 0.5)
    pts = points_dict.get(squad_name, 3)
    return 1.0 + (prob * 0.1) + (pts * 0.01)

df_data['team_multiplier'] = df_data['squad'].apply(get_team_multiplier)

print("\nData preprocessing completed!")
print(f"Team multiplier range: {df_data['team_multiplier'].min():.4f} to {df_data['team_multiplier'].max():.4f}")

Filled missing values for tkl
Filled missing values for tklw
Filled missing values for tkl%
Filled missing values for tkl+int
Filled missing values for min
Filled missing values for blocks_stats_defense
Filled missing values for int

Data preprocessing completed!
Team multiplier range: 1.0800 to 1.0800


C:\Users\vardh\AppData\Local\Temp\ipykernel_35088\178885997.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_data['team_multiplier'] = df_data['squad'].apply(get_team_multiplier)


## Section 4: Train Random Forest Model

Train a RandomForestRegressor using tackle statistics as features to predict tackles won (tklw), discovering accurate tackle win vectors.

In [4]:
# Train Random Forest Regressor to discover accurate tackle win vectors
features = ['tkl', 'tkl%', 'tkl+int', 'min', 'blocks_stats_defense', 'int']

# Filter to available features in the dataset
available_features = [f for f in features if f in df_data.columns]
print(f"Using features: {available_features}")

X = df_data[available_features].fillna(0)
y = df_data['tklw'].fillna(0)

# Remove any rows with NaN or infinite values
valid_idx = ~(X.isnull().any(axis=1) | y.isnull() | np.isinf(X).any(axis=1) | np.isinf(y))
X = X[valid_idx]
y = y[valid_idx]

print(f"Training set size: {len(X)} samples")

# Train the model
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

print("\nRandom Forest model trained successfully!")
print(f"Model R² score: {rf.score(X, y):.4f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': available_features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(feature_importance.to_string(index=False))

Using features: ['tkl', 'tkl%', 'tkl+int', 'min', 'blocks_stats_defense', 'int']
Training set size: 2854 samples

Random Forest model trained successfully!
Model R² score: 0.9945

Feature Importance:
             feature  importance
                 tkl    0.967932
                 min    0.008218
                tkl%    0.007326
blocks_stats_defense    0.006104
             tkl+int    0.006098
                 int    0.004323


## Section 5: Calculate Tackle Scores

Compute baseline scores from the Random Forest model and multiply by team context multipliers to create final tackle scores.

In [5]:
# Compute baseline profile score and map it with contextual factors
X_all = df_data[available_features].fillna(0)
df_data['base_score'] = rf.predict(X_all)
df_data['tackle_score'] = df_data['base_score'] * df_data['team_multiplier']

print("Tackle scores calculated!")
print(f"Base score range: {df_data['base_score'].min():.4f} to {df_data['base_score'].max():.4f}")
print(f"Final tackle score range: {df_data['tackle_score'].min():.4f} to {df_data['tackle_score'].max():.4f}")

# Show statistics
print("\nTackle Score Statistics:")
print(df_data['tackle_score'].describe())

Tackle scores calculated!
Base score range: 0.0000 to 77.1900
Final tackle score range: 0.0000 to 83.3652

Tackle Score Statistics:
count    2854.000000
mean       13.219433
std        13.641036
min         0.000000
25%         1.881900
50%         8.861400
75%        20.838600
max        83.365200
Name: tackle_score, dtype: float64


C:\Users\vardh\AppData\Local\Temp\ipykernel_35088\1775745184.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_data['base_score'] = rf.predict(X_all)
C:\Users\vardh\AppData\Local\Temp\ipykernel_35088\1775745184.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_data['tackle_score'] = df_data['base_score'] * df_data['team_multiplier']


## Section 6: Generate Best Tacklers Leaderboard

Create a formatted leaderboard with player rank, name, team, position, total tackles, tackles won, tackle win rate, and tackle score sorted in descending order.

In [6]:
# Final leaderboard format
required_columns = ['player', 'squad', 'pos', 'tkl', 'tklw', 'tkl%', 'tackle_score']
available_cols = [col for col in required_columns if col in df_data.columns]

leaderboard = df_data[available_cols].copy()
leaderboard = leaderboard.rename(columns={
    'squad': 'team', 'pos': 'position', 'tkl': 'total_tackles',
    'tklw': 'tackles_won', 'tkl%': 'tackle_win_rate_pct'
})

# Sort by tackle score descending
leaderboard = leaderboard.sort_values(by='tackle_score', ascending=False).reset_index(drop=True)
leaderboard['rank'] = leaderboard.index + 1

# Reorder columns
final_columns = ['rank', 'player', 'team', 'position', 'total_tackles', 'tackles_won', 'tackle_win_rate_pct', 'tackle_score']
leaderboard = leaderboard[[col for col in final_columns if col in leaderboard.columns]]

print("Best Tacklers Leaderboard Generated!")
print(f"Total players ranked: {len(leaderboard)}")
print(f"\nLeaderboard shape: {leaderboard.shape}")

Best Tacklers Leaderboard Generated!
Total players ranked: 2854

Leaderboard shape: (2854, 8)


## Section 7: Save Results to Player Stats

Export the final leaderboard to player_stats as a CSV file and display the top 20 players.

In [7]:
# Save the leaderboard to CSV file in player_stats directory
output_file = 'best_tacklers_leaderboard.csv'
leaderboard.to_csv(output_file, index=False)
print(f"✓ Leaderboard saved to: {output_file}")

# Display top 20 players
print("\n" + "="*100)
print("TOP 20 BEST TACKLERS")
print("="*100)
display_cols = ['rank', 'player', 'team', 'position', 'total_tackles', 'tackles_won', 'tackle_score']
top_20 = leaderboard[display_cols].head(20)
print(top_20.to_string(index=False))

# Additional statistics
print("\n" + "="*100)
print("LEADERBOARD STATISTICS")
print("="*100)
print(f"Total players ranked: {len(leaderboard)}")
print(f"Top scorer tackle score: {leaderboard['tackle_score'].max():.4f}")
print(f"Average tackle score: {leaderboard['tackle_score'].mean():.4f}")
print(f"Median tackle score: {leaderboard['tackle_score'].median():.4f}")

# Save summary statistics
stats_summary = {
    'Total Players': len(leaderboard),
    'Top Score': leaderboard['tackle_score'].max(),
    'Average Score': leaderboard['tackle_score'].mean(),
    'Median Score': leaderboard['tackle_score'].median(),
    'Min Score': leaderboard['tackle_score'].min(),
    'Max Score': leaderboard['tackle_score'].max()
}

print("\nProcessing Complete! ✓")

✓ Leaderboard saved to: best_tacklers_leaderboard.csv

TOP 20 BEST TACKLERS
 rank                player            team position  total_tackles  tackles_won  tackle_score
    1    Idrissa Gana Gueye         Everton       MF            133           80       83.3652
    2        Omar El Hilali        Espanyol       DF            111           79       83.2032
    3          Daniel Muñoz  Crystal Palace       DF            123           80       82.8036
    4        Moisés Caicedo         Chelsea    MF,DF            114           73       80.7948
    5         Andrey Santos      Strasbourg       MF            110           74       78.1488
    6            João Gomes          Wolves       MF            116           71       76.1832
    7     Noussair Mazraoui  Manchester Utd       DF            115           68       74.7252
    8          Jon Aramburu   Real Sociedad       DF            102           70       70.5888
    9       Morten Frendrup           Genoa       MF            102  